In [ ]:
# Fabric-native parameters and config
# Edit these values directly in Fabric, or pass them from a Fabric Data Pipeline.
ENV = "dev"
SCHEMA = "dbo"
MARKET = "AU"
MOCK_IPSTACK = True
LOOKBACK_DAYS = 90
LISTINGS_FILE = "listings.csv"
VIEWS_FILE = "property_views_5k.csv"
MAX_IPSTACK_CALLS = 5  # safety cap for live IPstack tests
FORCE_REFRESH_IPSTACK = False
KEY_VAULT_NAME = "kv-ipstack"  # pass from Fabric Pipeline when different
IPSTACK_SECRET_NAME = "ipstack-access-key"
IPSTACK_ACCESS_KEY = ""  # local fallback only; never commit a real key

try:
    from notebookutils import mssparkutils
except ImportError:
    mssparkutils = None

from pyspark.sql import SparkSession
spark = SparkSession.getActiveSession() or SparkSession.builder.getOrCreate()

class Config:
    def __init__(self):
        self.env = ENV
        self.schema = SCHEMA
        self.market = MARKET
        self.mock_ipstack = MOCK_IPSTACK
        self.lookback_days = LOOKBACK_DAYS
        self.listings_file = LISTINGS_FILE
        self.views_file = VIEWS_FILE
        self.max_ipstack_calls = MAX_IPSTACK_CALLS
        self.force_refresh_ipstack = FORCE_REFRESH_IPSTACK
        self.key_vault_name = KEY_VAULT_NAME
        self.ipstack_secret_name = IPSTACK_SECRET_NAME
        self.ipstack_access_key = IPSTACK_ACCESS_KEY
        self.files_base = "Files"  # Fabric attached Lakehouse path
        self.listings_path = f"{self.files_base}/raw/sample/{self.listings_file}"
        self.views_path = f"{self.files_base}/raw/sample/{self.views_file}"
        self.fixture_base = f"{self.files_base}/fixtures/ipstack"

        self.bronze_listings = "bronze_listings"
        self.bronze_property_views = "bronze_property_views"
        self.bronze_ipstack_raw = "bronze_ipstack_raw"
        self.bronze_ipstack_errors = "bronze_ipstack_errors"
        self.silver_listings = "silver_listings"
        self.silver_ip_dim = "silver_ip_dim"
        self.silver_visits_enriched = "silver_visits_enriched"
        self.gold_suburb_interest = "gold_suburb_interest"
        self.gold_property_type_by_suburb = "gold_property_type_by_suburb"
        self.gold_price_engagement = "gold_price_engagement"
        self.gold_conversion_gaps = "gold_conversion_gaps"
        self.gold_property_trends = "gold_property_trends"
        self.gold_region_type_preference = "gold_region_type_preference"
        self.gold_repeat_interest = "gold_repeat_interest"
        self.gold_interstate_flow = "gold_interstate_flow"

    def table_fqn(self, table: str) -> str:
        return f"{self.schema}.{table}"

conf = Config()
print(f"env={conf.env} market={conf.market} mock={conf.mock_ipstack} views={conf.views_file}")
print(f"listings_path={conf.listings_path}")
print(f"views_path={conf.views_path}")






In [ ]:
import json
import os
import time
import urllib.parse
import urllib.request
from pathlib import Path

from pyspark.sql import functions as F
from pyspark.sql.window import Window


class SilverLoader:
    def __init__(self):
        self.c = conf

    def distinct_ips(self):
        return [
            r.ip
            for r in spark.table(self.c.table_fqn(self.c.bronze_property_views))
            .select("ip")
            .distinct()
            .collect()
            if r.ip
        ]

    def reset_ipstack_cache(self):
        # Old runs appended Victoria fallback JSON to bronze_ipstack_raw and silver_ip_dim
        # cached those IPs — mock mode must start clean every time.
        if self.c.mock_ipstack or self.c.force_refresh_ipstack:
            for table in (self.c.bronze_ipstack_raw, self.c.silver_ip_dim):
                spark.sql(f"DROP TABLE IF EXISTS {self.c.table_fqn(table)}")
            print("Reset IPstack cache for clean enrichment")

    def already_enriched_ips(self):
        # Mock fixtures are local/dev — always re-enrich so stale cache cannot win.
        if self.c.force_refresh_ipstack or self.c.mock_ipstack:
            return set()
        try:
            if not spark.catalog.tableExists(self.c.table_fqn(self.c.silver_ip_dim)):
                return set()
            return {
                r.ip
                for r in spark.table(self.c.table_fqn(self.c.silver_ip_dim))
                .select("ip")
                .distinct()
                .collect()
                if r.ip
            }
        except Exception:
            return set()

    def ips_to_enrich(self, ips):
        existing = self.already_enriched_ips()
        pending = [ip for ip in ips if ip not in existing]
        if self.c.mock_ipstack:
            return pending
        capped = pending[: self.c.max_ipstack_calls]
        skipped = len(pending) - len(capped)
        if skipped > 0:
            print(
                f"Live IPstack safety cap: enriching {len(capped)} IPs, "
                f"skipping {skipped}. Increase MAX_IPSTACK_CALLS when ready."
            )
        return capped

    def load_mock(self, ip):
        rel_path = f"{self.c.fixture_base}/{ip}.json"
        candidates = [
            f"/lakehouse/default/{rel_path}",
            rel_path,
            f"data/fixtures/ipstack/{ip}.json",
        ]
        for p in candidates:
            if mssparkutils is not None:
                try:
                    if mssparkutils.fs.exists(p):
                        return mssparkutils.fs.head(p, 1_000_000), 200
                except Exception:
                    pass
            local = Path(p)
            if local.exists():
                return local.read_text(), 200
        raise FileNotFoundError(
            f"No IPstack fixture for {ip}. Upload {ip}.json to {self.c.fixture_base}/ in the Lakehouse."
        )

    def ipstack_key(self):
        if self.c.ipstack_access_key:
            return self.c.ipstack_access_key

        if self.c.key_vault_name and mssparkutils is not None:
            try:
                return mssparkutils.credentials.getSecret(
                    self.c.key_vault_name, self.c.ipstack_secret_name
                )
            except Exception:
                vault_uri = self.c.key_vault_name
                if not vault_uri.startswith("https://"):
                    vault_uri = f"https://{self.c.key_vault_name}.vault.azure.net/"
                return mssparkutils.credentials.getSecret(vault_uri, self.c.ipstack_secret_name)

        return os.getenv("IPSTACK_ACCESS_KEY", "")

    def load_live(self, ip):
        key = self.ipstack_key()
        if not key:
            raise ValueError("IPSTACK_ACCESS_KEY required when MOCK_IPSTACK=False")
        safe_ip = urllib.parse.quote(ip, safe="")
        safe_key = urllib.parse.quote(key, safe="")
        url = f"http://api.ipstack.com/{safe_ip}?access_key={safe_key}&security=1"
        with urllib.request.urlopen(url, timeout=15) as resp:
            return resp.read().decode(), resp.getcode()

    def write_ipstack_error(self, ip, error_message, status=0):
        err = spark.createDataFrame(
            [(ip, str(error_message), int(status))],
            "ip STRING, error_message STRING, http_status INT",
        )
        err.withColumn("api_called_at", F.current_timestamp()).write.format("delta").mode(
            "append"
        ).saveAsTable(self.c.table_fqn(self.c.bronze_ipstack_errors))

    def enrich_ips(self, ips):
        target_ips = self.ips_to_enrich(ips)
        print(
            f"IPstack target IPs: {len(target_ips)} "
            f"(distinct={len(ips)}, mock={self.c.mock_ipstack}, "
            f"force_refresh={self.c.force_refresh_ipstack})"
        )
        if not target_ips:
            print("No IPs to enrich — if this is unexpected, set FORCE_REFRESH_IPSTACK=True")
            return

        rows = []
        errors = []
        for ip in target_ips:
            try:
                body, status = self.load_mock(ip) if self.c.mock_ipstack else self.load_live(ip)
                if not self.c.mock_ipstack:
                    time.sleep(0.25)
                rows.append((ip, body, int(status)))
            except Exception as e:
                errors.append((ip, str(e)))
                self.write_ipstack_error(ip, e)

        if errors:
            print(f"Fixture/API errors for {len(errors)} IP(s):")
            for ip, msg in errors[:10]:
                print(f"  {ip}: {msg}")

        if rows:
            df = spark.createDataFrame(rows, "ip STRING, response_json STRING, http_status INT")
            write_mode = "overwrite" if self.c.mock_ipstack else "append"
            df.withColumn("api_called_at", F.current_timestamp()).write.format("delta").mode(
                write_mode
            ).saveAsTable(self.c.table_fqn(self.c.bronze_ipstack_raw))
            print(f"Wrote {len(rows)} IPstack responses ({write_mode})")

    def build_ip_dim(self):
        j = (
            "ip STRING, country_code STRING, country_name STRING, region_name STRING, "
            "city STRING, latitude DOUBLE, longitude DOUBLE, time_zone STRUCT<id:STRING>, "
            "connection STRUCT<isp:STRING>, "
            "security STRUCT<is_proxy:BOOLEAN,proxy_type:STRING>"
        )
        raw = spark.table(self.c.table_fqn(self.c.bronze_ipstack_raw))
        p = raw.withColumn("j", F.from_json("response_json", j))
        latest = Window.partitionBy("ip").orderBy(F.desc("api_called_at"))
        dim = (
            p.select(
                "ip",
                "api_called_at",
                F.col("j.country_code").alias("country_code"),
                F.col("j.country_name").alias("country_name"),
                F.col("j.region_name").alias("visitor_region"),
                F.col("j.city").alias("visitor_city"),
                F.col("j.latitude").alias("latitude"),
                F.col("j.longitude").alias("longitude"),
                F.col("j.time_zone.id").alias("timezone_id"),
                F.col("j.connection.isp").alias("isp"),
                F.when(F.col("j.security.proxy_type") == "vpn", True)
                .otherwise(F.coalesce(F.col("j.security.is_proxy"), F.lit(False)))
                .alias("is_vpn"),
            )
            .withColumn("rn", F.row_number().over(latest))
            .filter(F.col("rn") == 1)
            .drop("rn", "api_called_at")
            .withColumn("enriched_at", F.current_timestamp())
        )
        dim.write.format("delta").mode("overwrite").saveAsTable(self.c.table_fqn(self.c.silver_ip_dim))

    def build_listings(self):
        spark.table(self.c.table_fqn(self.c.bronze_listings)).drop("_ingested_at").write.format(
            "delta"
        ).mode("overwrite").saveAsTable(self.c.table_fqn(self.c.silver_listings))

    def build_visits(self):
        v = spark.table(self.c.table_fqn(self.c.bronze_property_views))
        l = spark.table(self.c.table_fqn(self.c.silver_listings))
        g = spark.table(self.c.table_fqn(self.c.silver_ip_dim))

        state_map = (
            F.when(F.col("visitor_region").contains("New South Wales"), "NSW")
            .when(F.col("visitor_region").contains("Victoria"), "VIC")
            .when(F.col("visitor_region").contains("Queensland"), "QLD")
            .when(F.col("visitor_region").contains("South Australia"), "SA")
            .when(F.col("visitor_region").contains("Western Australia"), "WA")
            .otherwise(F.lit("OTHER"))
        )
        device = F.when(F.col("user_agent").contains("iPhone"), F.lit("mobile")).otherwise(
            F.lit("desktop")
        )

        out = (
            v.join(l, "property_id", "inner")
            .join(g, "ip", "left")
            .withColumn("visitor_state", state_map)
            .withColumn("device_type", device)
            .withColumnRenamed("state", "listing_state")
            .select(
                "event_id",
                "session_id",
                "event_ts",
                "user_id",
                "ip",
                "property_id",
                "view_duration_sec",
                "enquiry_flag",
                "favorite_flag",
                "event_date",
                F.col("country_name").alias("visitor_country"),
                "visitor_region",
                "visitor_city",
                "visitor_state",
                F.col("latitude").alias("visitor_latitude"),
                F.col("longitude").alias("visitor_longitude"),
                "timezone_id",
                "isp",
                "is_vpn",
                "device_type",
                "suburb",
                "listing_state",
                "property_type",
                "price_range",
                "is_luxury",
                "price",
            )
        )
        out.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(
            self.c.table_fqn(self.c.silver_visits_enriched)
        )

    def validate_regions(self):
        dim = spark.table(self.c.table_fqn(self.c.silver_ip_dim))
        visits = spark.table(self.c.table_fqn(self.c.silver_visits_enriched))

        print("--- silver_ip_dim (one row per IP) ---")
        dim.select("ip", "visitor_region", "visitor_city").orderBy("ip").show(truncate=False)

        print("--- visitor region mix ---")
        visits.groupBy("visitor_country", "visitor_region", "visitor_state", "visitor_city").agg(
            F.count("*").alias("views"),
            F.countDistinct("ip").alias("distinct_ips"),
        ).orderBy(F.desc("views")).show(truncate=False)

        distinct_regions = dim.select("visitor_region").distinct().count()
        print(f"Distinct visitor regions in silver_ip_dim: {distinct_regions}")
        if distinct_regions <= 1:
            raise RuntimeError(
                "Only one visitor region detected. Upload fixtures to Files/fixtures/ipstack/ "
                "and re-run with MOCK_IPSTACK=True."
            )

    def run(self):
        self.reset_ipstack_cache()
        ips = self.distinct_ips()
        print(f"Enriching {len(ips)} visitor IPs (mock={self.c.mock_ipstack})")
        self.enrich_ips(ips)
        self.build_ip_dim()
        self.build_listings()
        self.build_visits()
        self.validate_regions()
        print("✅ Silver complete")


SilverLoader().run()





